Regenerate the six null-distribution panels from cached results, as separate SVGs.

Does NOT re-run the permutation (null_statistics.csv is read as-is). The only
thing recomputed is the observed-cluster statistics dict, which is cheap: it
reruns consensus_table() + realization_statistics() on the already-saved
Combined_meanRank.csv, with no KEA3 API calls and no shuffling.

Panel style is copied from kea3_permutation.plot_null so the output matches
the original combined figure, just one statistic per file.

In [1]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

from kea3_pipeline import cluster_membership
from kea3_consensus import consensus_table
from kea3_permutation import STATISTIC_TAILS, _empirical_p, realization_statistics

ROOT = "20260819-1231-s140206771-funny_keller"
HGNC_PATH = "hgnc_complete_set.txt"

LEIDEN_PATH = os.path.join(ROOT, "True-data-Leiden-clusters.csv")
COMBINED_MEANRANK = os.path.join(ROOT, "1000-iterations-v2", "kinase-enrichment", "Combined_meanRank.csv")
NULL_STATS_PATH = os.path.join(ROOT, "1000-iterations-v2", "kea-comparison", "perm_null", "null_statistics.csv")

OUTDIR = os.path.join(ROOT, "1000-iterations-v2", "figures")
os.makedirs(OUTDIR, exist_ok=True)

In [2]:
# --- observed statistics: cheap recompute from cached KEA3 output, no permutation re-run ---
membership = cluster_membership(
    LEIDEN_PATH, label_col="LeidenCluster", gene_col="Gene",
    minsize=20, maxsize=999, cluster_offset=1,
    hgnc_complete_set_path=HGNC_PATH,
)
consensus = consensus_table(COMBINED_MEANRANK)
observed = realization_statistics(
    consensus, top_n=12, min_n_top=4, background_frac=0.5, membership=membership,
)

In [3]:
# --- null distributions straight from the cached summary ---
null_df = pd.read_csv(NULL_STATS_PATH)

pvals = {
    statistic: _empirical_p(observed[statistic], null_df[statistic])
    for statistic in STATISTIC_TAILS
    if statistic in null_df.columns
}

In [4]:
# --- one SVG per statistic, same panel style as kea3_permutation.plot_null ---
for statistic in STATISTIC_TAILS:
    if statistic not in null_df.columns:
        continue
    values = null_df[statistic].dropna()
    if values.empty:
        continue

    fig, ax = plt.subplots(figsize=(5, 3.8))
    ax.hist(values, bins=25, alpha=0.75, label="Shuffled")
    ax.axvline(observed[statistic], lw=2, label=f"Observed = {observed[statistic]:.3f}")
    ax.set_title(f"{statistic}\ntwo-tailed p = {pvals[statistic]:.4f}", fontsize=10)
    ax.set_xlabel(statistic)
    ax.set_ylabel("Permutations")
    ax.legend(fontsize=8)
    fig.tight_layout()

    outfile = os.path.join(OUTDIR, f"{statistic}.svg")
    fig.savefig(outfile, format="svg")
    plt.close(fig)
    print(f"saved {outfile}  (observed={observed[statistic]:.3f}, p={pvals[statistic]:.4f})")

saved 20260819-1231-s140206771-funny_keller\1000-iterations-v2\figures\evidence_concordance.svg  (observed=0.171, p=0.0280)
saved 20260819-1231-s140206771-funny_keller\1000-iterations-v2\figures\mean_jaccard.svg  (observed=0.132, p=0.0020)
saved 20260819-1231-s140206771-funny_keller\1000-iterations-v2\figures\background_fraction.svg  (observed=0.283, p=0.0020)
saved 20260819-1231-s140206771-funny_keller\1000-iterations-v2\figures\frac_unique_kinases.svg  (observed=0.338, p=0.5095)
saved 20260819-1231-s140206771-funny_keller\1000-iterations-v2\figures\mean_call_strength.svg  (observed=6.647, p=0.6014)
saved 20260819-1231-s140206771-funny_keller\1000-iterations-v2\figures\self_membership_delta.svg  (observed=48.500, p=0.0999)
